In [5]:
# =============================================================================
# NEUROCORE — Indicizzazione dataset BIDS (IRCCS-ready)
# Inventario completo + sintesi per soggetto + report strutturato
# Output: /Volumes/KINGSTON/04920_Ecosystem/DATASETS/<dataset>/NEUROCORE_IRCCS_INDEX/
# =============================================================================

from pathlib import Path
import json, re, time
import pandas as pd

# ==============================
# PERCORSO CORRETTO (DA TE)
# ==============================
DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")

# ==============================
# VALIDAZIONE
# ==============================
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Directory dataset non trovata: {DATASET_DIR}")

sub_dirs = sorted([p for p in DATASET_DIR.glob("sub-*") if p.is_dir()])
if len(sub_dirs) == 0:
    raise RuntimeError(
        f"Nessuna cartella 'sub-*' trovata in:\n{DATASET_DIR}\n"
        "Verifica che questa sia la ROOT del dataset BIDS (dove stanno dataset_description.json e sub-...)."
    )

OUT_DIR = DATASET_DIR / "NEUROCORE_IRCCS_INDEX"
OUT_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

# ==============================
# FUNZIONI
# ==============================
def safe_read_json(p: Path):
    try:
        return json.loads(p.read_text(encoding="utf-8", errors="ignore"))
    except Exception:
        return None

def file_type(p: Path) -> str:
    suf = "".join(p.suffixes).lower()
    if suf.endswith(".nii") or suf.endswith(".nii.gz"): return "NIfTI"
    if suf.endswith(".edf"): return "EDF"
    if suf.endswith(".vhdr") or suf.endswith(".vmrk") or suf.endswith(".eeg"): return "BrainVision"
    if suf.endswith(".fif"): return "FIF"
    if suf.endswith(".tsv"): return "TSV"
    if suf.endswith(".json"): return "JSON"
    return suf.replace(".", "").upper() if suf else "FILE"

def modality_from_relpath(rel: str) -> str:
    s = rel.lower()
    if "/anat/" in s: return "anat"
    if "/func/" in s: return "func"
    if "/dwi/"  in s: return "dwi"
    if "/eeg/"  in s: return "eeg"
    if "/meg/"  in s: return "meg"
    if "/ieeg/" in s: return "ieeg"
    if "/beh/"  in s: return "beh"
    return ""

def parse_bids_entities(fname: str) -> dict:
    ent = {}
    for k, v in re.findall(r"([a-zA-Z]+)-([a-zA-Z0-9]+)", fname):
        ent[k.lower()] = v
    return ent

def uniq_join(series) -> str:
    vals = [str(v).strip() for v in series.dropna().tolist() if str(v).strip() != ""]
    return ", ".join(sorted(set(vals))) if vals else ""

# ==============================
# 1) METADATI ROOT
# ==============================
root_files = ["dataset_description.json", "README", "CHANGES", "participants.tsv", "participants.json"]
root_meta = {}
for name in root_files:
    p = DATASET_DIR / name
    if p.exists():
        if p.suffix.lower() == ".json":
            root_meta[name] = safe_read_json(p)
        else:
            root_meta[name] = {"presente": True, "dimensione_byte": p.stat().st_size}
    else:
        root_meta[name] = {"presente": False}

(OUT_DIR / "root_metadata.json").write_text(
    json.dumps(root_meta, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# ==============================
# 2) INVENTARIO COMPLETO
# ==============================
rows = []
total_bytes = 0

for sub in sub_dirs:
    for f in sub.rglob("*"):
        if f.is_file():
            sz = f.stat().st_size
            total_bytes += sz
            rel = f.relative_to(DATASET_DIR).as_posix()
            ent = parse_bids_entities(f.name)
            rows.append({
                "percorso_relativo": rel,
                "soggetto": ent.get("sub", ""),
                "sessione": ent.get("ses", ""),
                "task": ent.get("task", ""),
                "run": ent.get("run", ""),
                "modality_cartella": modality_from_relpath("/" + rel),
                "tipo_file": file_type(f),
                "dimensione_MB": round(sz / (1024**2), 3),
                "nome_file": f.name,
            })

inv = pd.DataFrame(rows)
if inv.empty:
    raise RuntimeError("Inventario vuoto: nessun file trovato sotto 'sub-*'.")

inv.to_csv(OUT_DIR / "inventory_full.csv", index=False)

# ==============================
# 3) SINTESI PER SOGGETTO
# ==============================
sub_summary = (
    inv.groupby("soggetto", dropna=False)
      .agg(
          n_file=("percorso_relativo", "count"),
          dimensione_totale_MB=("dimensione_MB", "sum"),
          modality_cartella=("modality_cartella", uniq_join),
          tipo_file=("tipo_file", uniq_join),
          task=("task", uniq_join),
          sessione=("sessione", uniq_join),
      )
      .reset_index()
)

sub_summary["dimensione_totale_MB"] = sub_summary["dimensione_totale_MB"].round(2)
sub_summary = sub_summary.sort_values(["soggetto"])
sub_summary.to_csv(OUT_DIR / "summary_by_subject.csv", index=False)

# ==============================
# 4) SINTESI GLOBALE + REPORT
# ==============================
global_summary = {
    "dataset_dir": str(DATASET_DIR),
    "output_dir": str(OUT_DIR),
    "n_soggetti": int(sub_summary["soggetto"].nunique()),
    "n_file_totali": int(len(inv)),
    "dimensione_totale_GB_stimata": round(total_bytes / (1024**3), 3),
    "modality_cartella_presenti": sorted(set([m for m in inv["modality_cartella"].tolist() if m])),
    "tipi_file_presenti": sorted(set(inv["tipo_file"].tolist())),
    "task_presenti": sorted(set([t for t in inv["task"].tolist() if str(t).strip() != ""])),
}

(OUT_DIR / "global_summary.json").write_text(
    json.dumps(global_summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

elapsed = time.time() - t0
report_lines = [
    "NEUROCORE — Report di indicizzazione dataset (BIDS)",
    "=" * 62,
    f"Directory dataset: {DATASET_DIR}",
    f"Directory output:  {OUT_DIR}",
    "",
    "Sintesi globale",
    "-" * 62,
    f"Soggetti individuati: {global_summary['n_soggetti']}",
    f"File indicizzati:     {global_summary['n_file_totali']}",
    f"Dimensione stimata:   {global_summary['dimensione_totale_GB_stimata']} GB",
    f"Modality (da cartelle): {', '.join(global_summary['modality_cartella_presenti']) or 'n/d'}",
    f"Tipi file:              {', '.join(global_summary['tipi_file_presenti'])}",
    f"Task (da filename):     {', '.join(global_summary['task_presenti']) or 'n/d'}",
    "",
    "File prodotti",
    "-" * 62,
    "1) inventory_full.csv         — inventario completo (file-level)",
    "2) summary_by_subject.csv     — sintesi per soggetto (subject-level)",
    "3) root_metadata.json         — metadati root BIDS",
    "4) global_summary.json        — riepilogo globale strutturato",
    "",
    f"Tempo esecuzione: {elapsed:.2f} secondi",
]
(OUT_DIR / "REPORT_IRCCS.txt").write_text("\n".join(report_lines), encoding="utf-8")

print("\n".join(report_lines))
display(sub_summary.head(50))

NEUROCORE — Report di indicizzazione dataset (BIDS)
Directory dataset: /Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia
Directory output:  /Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia/NEUROCORE_IRCCS_INDEX

Sintesi globale
--------------------------------------------------------------
Soggetti individuati: 3
File indicizzati:     168
Dimensione stimata:   1.086 GB
Modality (da cartelle): eeg
Tipi file:              FDT, FILE, JSON, SET, TSV
Task (da filename):     PictureNaming

File prodotti
--------------------------------------------------------------
1) inventory_full.csv         — inventario completo (file-level)
2) summary_by_subject.csv     — sintesi per soggetto (subject-level)
3) root_metadata.json         — metadati root BIDS
4) global_summary.json        — riepilogo globale strutturato

Tempo esecuzione: 0.28 secondi


,soggetto,n_file,dimensione_totale_MB,modality_cartella,tipo_file,task,sessione
0,,28,0.11,,FILE,,"1, 12, 2, 20, 4, 6, 8"
1,G01,70,551.40,eeg,"FDT, JSON, SET, TSV",PictureNaming,"1, 12, 2, 20, 4, 6, 8"
2,G02,70,560.75,eeg,"FDT, JSON, SET, TSV",PictureNaming,"1, 12, 2, 20, 4, 6, 8"


In [7]:
# =============================================================================
# NEUROCORE — Analisi EEG (EEGLAB .set) — versione compatibile (fix SciPy uint16_codec)
# Output: <DATASET_DIR>/NEUROCORE_ANALYSIS_IRCCS/
# =============================================================================

from pathlib import Path
import re, json, time, math
import numpy as np
import pandas as pd

# ----------------------------
# CONFIG
# ----------------------------
DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")
OUT_DIR = DATASET_DIR / "NEUROCORE_ANALYSIS_IRCCS"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESAMPLE_HZ = 250
HP_HZ, LP_HZ = 1.0, 40.0
NOTCH_HZ = 50
REF = "average"

WIN_S = 4.0
STEP_S = 1.0
N_STATES = 4

BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta":  (12, 30),
    "gamma": (30, 40)
}

t0 = time.time()

# ----------------------------
# FIX COMPATIBILITÀ: SciPy loadmat (uint16_codec)
# ----------------------------
import scipy.io as sio

_ORIG_LOADMAT = sio.loadmat

def _loadmat_compat(*args, **kwargs):
    # MNE può passare 'uint16_codec' ma alcune versioni SciPy non lo supportano.
    kwargs.pop("uint16_codec", None)
    return _ORIG_LOADMAT(*args, **kwargs)

# Patch sia scipy.io.loadmat che eventuali alias interni
sio.loadmat = _loadmat_compat
try:
    import scipy.io.matlab as sio_matlab
    sio_matlab.loadmat = _loadmat_compat
except Exception:
    pass

# ----------------------------
# Imports MNE (dopo patch)
# ----------------------------
import mne
from mne.time_frequency import psd_array_welch

# ----------------------------
# Utility
# ----------------------------
def parse_bids_entities(name: str):
    ent = {}
    for k, v in re.findall(r"([a-zA-Z]+)-([a-zA-Z0-9]+)", name):
        ent[k.lower()] = v
    return ent

def shannon_entropy(p_vec, eps=1e-12):
    p = np.asarray(p_vec, dtype=float)
    p = p / (p.sum() + eps)
    p = np.clip(p, eps, 1.0)
    return float(-(p * np.log(p)).sum())

def kmeans_simple(X, k, n_iter=30, seed=0):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    k = min(max(1, k), n)
    idx = rng.choice(n, size=k, replace=False)
    C = X[idx].copy()
    for _ in range(n_iter):
        d2 = ((X[:, None, :] - C[None, :, :])**2).sum(axis=2)
        lab = d2.argmin(axis=1)
        C_new = C.copy()
        for j in range(k):
            sel = (lab == j)
            if sel.any():
                C_new[j] = X[sel].mean(axis=0)
        if np.allclose(C, C_new, atol=1e-6):
            break
        C = C_new
    return lab, C

def sliding_windows(n_samples, sfreq, win_s, step_s):
    win = int(round(win_s * sfreq))
    step = int(round(step_s * sfreq))
    starts = np.arange(0, max(0, n_samples - win + 1), step, dtype=int)
    return starts, win

def bandpower_from_psd(psd, freqs, band):
    fmin, fmax = band
    m = (freqs >= fmin) & (freqs < fmax)
    if not np.any(m):
        return np.zeros(psd.shape[0], dtype=float)
    df = float(np.mean(np.diff(freqs)))
    return psd[:, m].sum(axis=1) * df

# ----------------------------
# 1) Input .set
# ----------------------------
set_files = sorted([p for p in DATASET_DIR.rglob("*.set") if "sub-" in p.as_posix()])
if len(set_files) == 0:
    raise RuntimeError("Nessun file .set trovato. Verifica il dataset o il percorso DATASET_DIR.")

pd.DataFrame({"set_file": [p.relative_to(DATASET_DIR).as_posix() for p in set_files]}).to_csv(
    OUT_DIR / "inputs_set_files.csv", index=False
)

# ----------------------------
# 2) Analisi per file
# ----------------------------
per_file_rows = []
per_window_rows = []

for set_path in set_files:
    rel = set_path.relative_to(DATASET_DIR).as_posix()
    ent = parse_bids_entities(set_path.name)
    subject = ent.get("sub", "")
    ses = ent.get("ses", "")
    task = ent.get("task", "")
    run = ent.get("run", "")

    # Caricamento EEGLAB (ora compatibile)
    raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose="ERROR")

    # Selezione EEG e normalizzazione
    raw.pick_types(eeg=True, eog=True, ecg=True, emg=True, stim=False, misc=True)
    raw.set_eeg_reference(REF, projection=False, verbose="ERROR")

    # Filtri robusti
    try:
        raw.notch_filter(NOTCH_HZ, verbose="ERROR")
    except Exception:
        pass
    raw.filter(HP_HZ, LP_HZ, verbose="ERROR")
    raw.resample(RESAMPLE_HZ, npad="auto", verbose="ERROR")

    sfreq = float(raw.info["sfreq"])
    eeg = raw.copy().pick("eeg")
    data = eeg.get_data()  # (n_ch, n_samples)
    ch_names = eeg.ch_names

    if data.size == 0:
        continue

    n_ch, n_samp = data.shape
    duration_s = n_samp / sfreq

    # PSD globale (bande + entropia spaziale)
    psd, freqs = psd_array_welch(
        data, sfreq=sfreq, fmin=HP_HZ, fmax=LP_HZ,
        n_fft=int(round(sfreq * 4)), n_overlap=int(round(sfreq * 2)),
        verbose=False
    )
    total_power = psd.sum(axis=1)
    total_norm = total_power / (total_power.sum() + 1e-12)

    alpha = bandpower_from_psd(psd, freqs, BANDS["alpha"])
    alpha_norm = alpha / (alpha.sum() + 1e-12)
    global_alpha_ratio = float(alpha.sum() / (total_power.sum() + 1e-12))

    # Dinamica: finestre scorrevoli + clustering stati su topografia (theta+alpha)
    starts, win = sliding_windows(n_samp, sfreq, WIN_S, STEP_S)
    if len(starts) < 3:
        per_file_rows.append({
            "relative_path": rel,
            "subject": subject, "session": ses, "task": task, "run": run,
            "n_channels_eeg": int(n_ch),
            "sfreq_hz": float(sfreq),
            "duration_s": round(float(duration_s), 3),
            "global_alpha_ratio": global_alpha_ratio,
            "spatial_entropy_total_power": shannon_entropy(total_norm),
            "spatial_entropy_alpha_power": shannon_entropy(alpha_norm),
            "stabilita_organizzazione_funzione_media": np.nan,
            "transizioni_stato_n": np.nan,
            "transizioni_stato_rate_min": np.nan,
            "persistenza_temporale_mediana_s": np.nan,
            "indeterminatezza_funzione_entropy": np.nan,
            "resilienza_tempo_recupero_mediano_s": np.nan,
        })
        continue

    X = []
    win_times = []
    for s0 in starts:
        seg = data[:, s0:s0+win]
        psd_w, freqs_w = psd_array_welch(
            seg, sfreq=sfreq, fmin=HP_HZ, fmax=LP_HZ,
            n_fft=int(round(sfreq * 2)), n_overlap=int(round(sfreq * 1)),
            verbose=False
        )
        theta_w = bandpower_from_psd(psd_w, freqs_w, BANDS["theta"])
        alpha_w = bandpower_from_psd(psd_w, freqs_w, BANDS["alpha"])
        feat = theta_w + alpha_w
        feat = feat / (feat.sum() + 1e-12)
        X.append(feat)
        win_times.append(s0 / sfreq)

    X = np.vstack(X)
    win_times = np.asarray(win_times)

    labels, _ = kmeans_simple(X, k=N_STATES, n_iter=40, seed=42)

    # Stabilità: correlazione topografica tra finestre consecutive
    corr_seq = []
    for a in range(len(X) - 1):
        c = np.corrcoef(X[a], X[a+1])[0, 1]
        corr_seq.append(c if np.isfinite(c) else np.nan)
    corr_seq = np.asarray(corr_seq)
    stability_mean = float(np.nanmean(corr_seq))

    # Transizioni
    transitions = int(np.sum(labels[1:] != labels[:-1]))
    transitions_rate_min = float(transitions / (duration_s/60.0 + 1e-12))

    # Persistenza (run-length)
    runs = []
    run_len = 1
    for a in range(1, len(labels)):
        if labels[a] == labels[a-1]:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    runs = np.asarray(runs, dtype=float)
    persistence_median_s = float(np.median(runs) * STEP_S)

    # Indeterminatezza (entropia occupazione stati)
    occ = np.bincount(labels, minlength=int(labels.max())+1).astype(float)
    state_entropy = shannon_entropy(occ)

    # Resilienza (proxy): durata run successiva mediana
    recovery_times = (runs[1:] * STEP_S) if len(runs) > 1 else np.array([np.nan])
    recovery_time_median_s = float(np.nanmedian(recovery_times))

    # Audit trail per finestra
    topo_sim_next = np.r_[corr_seq, np.nan]
    for t_sec, lab, cval in zip(win_times, labels, topo_sim_next):
        per_window_rows.append({
            "relative_path": rel,
            "subject": subject, "session": ses, "task": task, "run": run,
            "t_window_start_s": round(float(t_sec), 3),
            "state_label": int(lab),
            "topo_similarity_next": float(cval) if np.isfinite(cval) else np.nan
        })

    per_file_rows.append({
        "relative_path": rel,
        "subject": subject, "session": ses, "task": task, "run": run,
        "n_channels_eeg": int(n_ch),
        "sfreq_hz": float(sfreq),
        "duration_s": round(float(duration_s), 3),
        "global_alpha_ratio": global_alpha_ratio,
        "spatial_entropy_total_power": shannon_entropy(total_norm),
        "spatial_entropy_alpha_power": shannon_entropy(alpha_norm),
        "stabilita_organizzazione_funzione_media": stability_mean,
        "transizioni_stato_n": transitions,
        "transizioni_stato_rate_min": transitions_rate_min,
        "persistenza_temporale_mediana_s": persistence_median_s,
        "indeterminatezza_funzione_entropy": state_entropy,
        "resilienza_tempo_recupero_mediano_s": recovery_time_median_s,
    })

# ----------------------------
# 3) Output
# ----------------------------
df_files = pd.DataFrame(per_file_rows)
df_windows = pd.DataFrame(per_window_rows)

df_files.to_csv(OUT_DIR / "NEUROCORE_metrics_per_file.csv", index=False)
df_windows.to_csv(OUT_DIR / "NEUROCORE_state_dynamics_windows.csv", index=False)

if not df_files.empty:
    df_sub = (df_files.groupby(["subject"], dropna=False)
              .agg(
                  n_recordings=("relative_path", "count"),
                  duration_s_mean=("duration_s", "mean"),
                  stabilita_organizzazione_funzione_media=("stabilita_organizzazione_funzione_media", "mean"),
                  transizioni_stato_rate_min=("transizioni_stato_rate_min", "mean"),
                  persistenza_temporale_mediana_s=("persistenza_temporale_mediana_s", "median"),
                  indeterminatezza_funzione_entropy=("indeterminatezza_funzione_entropy", "mean"),
                  resilienza_tempo_recupero_mediano_s=("resilienza_tempo_recupero_mediano_s", "median"),
                  global_alpha_ratio=("global_alpha_ratio", "mean"),
                  spatial_entropy_total_power=("spatial_entropy_total_power", "mean"),
              )
              .reset_index())
    df_sub.to_csv(OUT_DIR / "NEUROCORE_summary_by_subject.csv", index=False)
else:
    df_sub = pd.DataFrame()

elapsed = time.time() - t0

report_lines = [
    "NEUROCORE — Report analisi EEG (EEGLAB .set) — IRCCS-ready",
    "=" * 72,
    f"Directory dataset: {DATASET_DIR}",
    f"Directory output:  {OUT_DIR}",
    "",
    "Parametri principali",
    "-" * 72,
    f"Resample: {RESAMPLE_HZ} Hz | Filtri: {HP_HZ}-{LP_HZ} Hz | Notch: {NOTCH_HZ} Hz | Ref: {REF}",
    f"Finestre: WIN={WIN_S}s, STEP={STEP_S}s | N stati={N_STATES}",
    "",
    "Output prodotti",
    "-" * 72,
    "1) NEUROCORE_metrics_per_file.csv        — metriche per registrazione",
    "2) NEUROCORE_state_dynamics_windows.csv  — dinamica finestre/stati (audit trail)",
    "3) NEUROCORE_summary_by_subject.csv      — sintesi per soggetto",
    "",
    f"File .set analizzati: {len(set_files)}",
    f"Tempo esecuzione: {elapsed:.2f} secondi",
]
(OUT_DIR / "REPORT_ANALYSIS_IRCCS.txt").write_text("\n".join(report_lines), encoding="utf-8")

print("\n".join(report_lines))
display(df_files.head(30))
display(df_sub)

TypeError: MatFileReader.__init__() got an unexpected keyword argument 'uint16_codec'

In [8]:
# =============================================================================
# FIX IMMEDIATO — errore SciPy: MatFileReader.__init__ got unexpected keyword 'uint16_codec'
# Causa: MNE passa 'uint16_codec' a SciPy, ma la tua versione SciPy non lo supporta.
# Soluzione: patch a livello "mat_reader_factory" + MatFile4/5Reader per ignorare 'uint16_codec'.
# =============================================================================

from pathlib import Path
import re, json, time
import numpy as np
import pandas as pd

# ----------------------------
# PATCH SciPy (PRIMA di importare mne)
# ----------------------------
import scipy.io.matlab._mio as _mio
import scipy.io.matlab._mio4 as _mio4
import scipy.io.matlab._mio5 as _mio5

# 1) patch factory (dove nasce l'errore nello stacktrace)
_ORIG_MRF = _mio.mat_reader_factory
def _mat_reader_factory_compat(f, **kwargs):
    kwargs.pop("uint16_codec", None)
    return _ORIG_MRF(f, **kwargs)
_mio.mat_reader_factory = _mat_reader_factory_compat

# 2) patch reader init (rete di sicurezza)
_ORIG_INIT4 = _mio4.MatFile4Reader.__init__
def _init4_compat(self, *args, **kwargs):
    kwargs.pop("uint16_codec", None)
    return _ORIG_INIT4(self, *args, **kwargs)
_mio4.MatFile4Reader.__init__ = _init4_compat

_ORIG_INIT5 = _mio5.MatFile5Reader.__init__
def _init5_compat(self, *args, **kwargs):
    kwargs.pop("uint16_codec", None)
    return _ORIG_INIT5(self, *args, **kwargs)
_mio5.MatFile5Reader.__init__ = _init5_compat

# ----------------------------
# Import MNE (DOPO patch)
# ----------------------------
import mne
from mne.time_frequency import psd_array_welch

# =============================================================================
# NEUROCORE — Analisi EEG (EEGLAB .set) — IRCCS-ready
# Output: <DATASET_DIR>/NEUROCORE_ANALYSIS_IRCCS/
# =============================================================================

DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")
OUT_DIR = DATASET_DIR / "NEUROCORE_ANALYSIS_IRCCS"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Preprocessing robusto
RESAMPLE_HZ = 250
HP_HZ, LP_HZ = 1.0, 40.0
NOTCH_HZ = 50
REF = "average"

# Dinamica
WIN_S = 4.0
STEP_S = 1.0
N_STATES = 4

BANDS = {"theta": (4, 8), "alpha": (8, 12)}

t0 = time.time()

def parse_bids_entities(name: str):
    ent = {}
    for k, v in re.findall(r"([a-zA-Z]+)-([a-zA-Z0-9]+)", name):
        ent[k.lower()] = v
    return ent

def shannon_entropy(p_vec, eps=1e-12):
    p = np.asarray(p_vec, dtype=float)
    p = p / (p.sum() + eps)
    p = np.clip(p, eps, 1.0)
    return float(-(p * np.log(p)).sum())

def kmeans_simple(X, k, n_iter=30, seed=0):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    k = min(max(1, k), n)
    C = X[rng.choice(n, size=k, replace=False)].copy()
    for _ in range(n_iter):
        d2 = ((X[:, None, :] - C[None, :, :])**2).sum(axis=2)
        lab = d2.argmin(axis=1)
        C_new = C.copy()
        for j in range(k):
            sel = (lab == j)
            if sel.any():
                C_new[j] = X[sel].mean(axis=0)
        if np.allclose(C, C_new, atol=1e-6):
            break
        C = C_new
    return lab, C

def sliding_windows(n_samples, sfreq, win_s, step_s):
    win = int(round(win_s * sfreq))
    step = int(round(step_s * sfreq))
    starts = np.arange(0, max(0, n_samples - win + 1), step, dtype=int)
    return starts, win

def bandpower_from_psd(psd, freqs, band):
    fmin, fmax = band
    m = (freqs >= fmin) & (freqs < fmax)
    if not np.any(m):
        return np.zeros(psd.shape[0], dtype=float)
    df = float(np.mean(np.diff(freqs)))
    return psd[:, m].sum(axis=1) * df

set_files = sorted([p for p in DATASET_DIR.rglob("*.set") if "sub-" in p.as_posix()])
if not set_files:
    raise RuntimeError("Nessun file .set trovato. Verifica DATASET_DIR o la struttura del dataset.")

pd.DataFrame({"set_file": [p.relative_to(DATASET_DIR).as_posix() for p in set_files]}).to_csv(
    OUT_DIR / "inputs_set_files.csv", index=False
)

per_file_rows = []
per_window_rows = []

for set_path in set_files:
    rel = set_path.relative_to(DATASET_DIR).as_posix()
    ent = parse_bids_entities(set_path.name)
    subject = ent.get("sub", "")
    ses = ent.get("ses", "")
    task = ent.get("task", "")
    run = ent.get("run", "")

    raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose="ERROR")
    raw.pick_types(eeg=True, eog=True, ecg=True, emg=True, stim=False, misc=True)
    raw.set_eeg_reference(REF, projection=False, verbose="ERROR")

    try:
        raw.notch_filter(NOTCH_HZ, verbose="ERROR")
    except Exception:
        pass
    raw.filter(HP_HZ, LP_HZ, verbose="ERROR")
    raw.resample(RESAMPLE_HZ, npad="auto", verbose="ERROR")

    eeg = raw.copy().pick("eeg")
    data = eeg.get_data()
    if data.size == 0:
        continue

    sfreq = float(eeg.info["sfreq"])
    n_ch, n_samp = data.shape
    duration_s = n_samp / sfreq

    # PSD globale (alpha ratio + entropia spaziale totale)
    psd, freqs = psd_array_welch(
        data, sfreq=sfreq, fmin=HP_HZ, fmax=LP_HZ,
        n_fft=int(round(sfreq * 4)), n_overlap=int(round(sfreq * 2)),
        verbose=False
    )
    total_power = psd.sum(axis=1)
    total_norm = total_power / (total_power.sum() + 1e-12)

    alpha = bandpower_from_psd(psd, freqs, BANDS["alpha"])
    alpha_ratio = float(alpha.sum() / (total_power.sum() + 1e-12))

    # Dinamica su finestre (theta+alpha topografia normalizzata)
    starts, win = sliding_windows(n_samp, sfreq, WIN_S, STEP_S)
    if len(starts) < 3:
        per_file_rows.append({
            "relative_path": rel, "subject": subject, "session": ses, "task": task, "run": run,
            "n_channels_eeg": int(n_ch), "sfreq_hz": sfreq, "duration_s": round(duration_s, 3),
            "global_alpha_ratio": alpha_ratio,
            "spatial_entropy_total_power": shannon_entropy(total_norm),
            "stabilita_organizzazione_funzione_media": np.nan,
            "transizioni_stato_n": np.nan,
            "persistenza_temporale_mediana_s": np.nan,
            "indeterminatezza_funzione_entropy": np.nan,
        })
        continue

    X, win_times = [], []
    for s0 in starts:
        seg = data[:, s0:s0+win]
        psd_w, freqs_w = psd_array_welch(
            seg, sfreq=sfreq, fmin=HP_HZ, fmax=LP_HZ,
            n_fft=int(round(sfreq * 2)), n_overlap=int(round(sfreq * 1)),
            verbose=False
        )
        theta_w = bandpower_from_psd(psd_w, freqs_w, BANDS["theta"])
        alpha_w = bandpower_from_psd(psd_w, freqs_w, BANDS["alpha"])
        topo = (theta_w + alpha_w)
        topo = topo / (topo.sum() + 1e-12)
        X.append(topo)
        win_times.append(s0 / sfreq)

    X = np.vstack(X)
    win_times = np.asarray(win_times)

    labels, _ = kmeans_simple(X, k=N_STATES, n_iter=40, seed=42)

    # Stabilità: correlazione topografica tra finestre consecutive
    corr_seq = []
    for a in range(len(X) - 1):
        c = np.corrcoef(X[a], X[a+1])[0, 1]
        corr_seq.append(c if np.isfinite(c) else np.nan)
    corr_seq = np.asarray(corr_seq)
    stability_mean = float(np.nanmean(corr_seq))

    # Transizioni
    transitions = int(np.sum(labels[1:] != labels[:-1]))

    # Persistenza (run-length)
    runs, r = [], 1
    for a in range(1, len(labels)):
        if labels[a] == labels[a-1]:
            r += 1
        else:
            runs.append(r); r = 1
    runs.append(r)
    runs = np.asarray(runs, dtype=float)
    persistence_median_s = float(np.median(runs) * STEP_S)

    # Indeterminatezza (entropia occupazione stati)
    occ = np.bincount(labels, minlength=int(labels.max())+1).astype(float)
    state_entropy = shannon_entropy(occ)

    # Audit trail
    topo_sim_next = np.r_[corr_seq, np.nan]
    for t_sec, lab, cval in zip(win_times, labels, topo_sim_next):
        per_window_rows.append({
            "relative_path": rel, "subject": subject, "session": ses, "task": task, "run": run,
            "t_window_start_s": round(float(t_sec), 3),
            "state_label": int(lab),
            "topo_similarity_next": float(cval) if np.isfinite(cval) else np.nan
        })

    per_file_rows.append({
        "relative_path": rel, "subject": subject, "session": ses, "task": task, "run": run,
        "n_channels_eeg": int(n_ch), "sfreq_hz": sfreq, "duration_s": round(duration_s, 3),
        "global_alpha_ratio": alpha_ratio,
        "spatial_entropy_total_power": shannon_entropy(total_norm),
        "stabilita_organizzazione_funzione_media": stability_mean,
        "transizioni_stato_n": transitions,
        "persistenza_temporale_mediana_s": persistence_median_s,
        "indeterminatezza_funzione_entropy": state_entropy,
    })

df_files = pd.DataFrame(per_file_rows)
df_windows = pd.DataFrame(per_window_rows)

df_files.to_csv(OUT_DIR / "NEUROCORE_metrics_per_file.csv", index=False)
df_windows.to_csv(OUT_DIR / "NEUROCORE_state_dynamics_windows.csv", index=False)

df_sub = (df_files.groupby("subject", dropna=False)
          .agg(
              n_recordings=("relative_path", "count"),
              duration_s_mean=("duration_s", "mean"),
              stabilita_organizzazione_funzione_media=("stabilita_organizzazione_funzione_media", "mean"),
              transizioni_stato_n=("transizioni_stato_n", "sum"),
              persistenza_temporale_mediana_s=("persistenza_temporale_mediana_s", "median"),
              indeterminatezza_funzione_entropy=("indeterminatezza_funzione_entropy", "mean"),
              global_alpha_ratio=("global_alpha_ratio", "mean"),
          )
          .reset_index())
df_sub.to_csv(OUT_DIR / "NEUROCORE_summary_by_subject.csv", index=False)

elapsed = time.time() - t0
report = "\n".join([
    "NEUROCORE — Report analisi EEG (EEGLAB .set) — IRCCS-ready",
    "=" * 72,
    f"Directory dataset: {DATASET_DIR}",
    f"Directory output:  {OUT_DIR}",
    "",
    "Output prodotti",
    "-" * 72,
    "1) NEUROCORE_metrics_per_file.csv        — metriche per registrazione",
    "2) NEUROCORE_state_dynamics_windows.csv  — dinamica finestre/stati (audit trail)",
    "3) NEUROCORE_summary_by_subject.csv      — sintesi per soggetto",
    "",
    f"File .set analizzati: {len(set_files)}",
    f"Tempo esecuzione: {elapsed:.2f} secondi",
])
(OUT_DIR / "REPORT_ANALYSIS_IRCCS.txt").write_text(report, encoding="utf-8")

print(report)
display(df_files.head(30))
display(df_sub)

ValueError: Mat 4 mopt wrong format, byteswapping problem?

In [9]:
from pathlib import Path

p = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")  # dataset dir
set_files = sorted(p.rglob("*.set"))
print("N .set:", len(set_files))
print("Esempio:", set_files[0] if set_files else None)

def sniff_header(fp, n=256):
    b = fp.read_bytes()[:n]
    return b

if set_files:
    fp = set_files[0]
    h = sniff_header(fp, 256)
    print("\n--- HEADER (primi 128 bytes) ---")
    print(h[:128])
    print("\n--- CLASSIFICAZIONE ---")
    if h.startswith(b"MATLAB 5.0 MAT-file"):
        print("MAT v5 (classico) -> SciPy/MNE dovrebbe leggerlo.")
    elif h.startswith(b"\x89HDF\r\n\x1a\n") or h[:3] == b"HDF":
        print("MAT v7.3 (HDF5) -> SciPy NON lo legge. Serve h5py/eeglabio o fallback.")
    elif h.lstrip().startswith(b"{") or b"EEGLAB" in h[:200]:
        print("Sembra testo/altro wrapper -> non è il .set binario MAT standard.")
    else:
        print("Formato non riconosciuto -> possibile file corrotto o non-.set reale.")


N .set: 28
Esempio: /Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia/sub-G01/ses-1/eeg/._sub-G01_ses-1_task-PictureNaming_run-1_eeg.set

--- HEADER (primi 128 bytes) ---
b'\x00\x05\x16\x07\x00\x02\x00\x00Mac OS X        \x00\x02\x00\x00\x00\t\x00\x00\x002\x00\x00\x0e\xb0\x00\x00\x00\x02\x00\x00\x0e\xe2\x00\x00\x01\x1e\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00ATTR\x00\x00\x00\x01\x00\x00\x0e\xe2\x00\x00\x00\xc8\x00\x00\x00\xc7\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\xc8\x00\x00\x00\x15'

--- CLASSIFICAZIONE ---
Formato non riconosciuto -> possibile file corrotto o non-.set reale.


In [10]:
from pathlib import Path
import subprocess, sys

DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")

def is_hdf5(fp: Path) -> bool:
    h = fp.read_bytes()[:16]
    return h.startswith(b"\x89HDF\r\n\x1a\n") or h[:3] == b"HDF"

def pip_install(pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + pkgs
    print("Installing:", " ".join(pkgs))
    subprocess.check_call(cmd)

def read_eeglab_robust(set_path: Path):
    import mne
    try:
        return mne.io.read_raw_eeglab(set_path, preload=True, verbose="ERROR")
    except Exception as e1:
        msg = str(e1)
        if is_hdf5(set_path):
            print(f"[HDF5/v7.3] {set_path.name} -> serve h5py + eeglabio")
            try:
                # installa dipendenze per MAT v7.3
                pip_install(["h5py", "eeglabio"])
            except Exception as ie:
                raise RuntimeError(
                    "Install fallita. Alternativa: conda install -c conda-forge h5py eeglabio"
                ) from ie

            import importlib
            importlib.invalidate_caches()
            import mne
            # riprova dopo install
            return mne.io.read_raw_eeglab(set_path, preload=True, verbose="ERROR")

        # se NON è HDF5, allora è tipicamente corrotto o non-.set
        raise RuntimeError(
            "Il .set NON risulta HDF5, ma SciPy/MNE lo interpreta come MAT v4 e fallisce.\n"
            "Probabile: file corrotto/download incompleto oppure .set non-EEGLAB reale.\n"
            f"Errore originale: {msg}"
        ) from e1

# Test singolo file
set_files = sorted(DATASET_DIR.rglob("*.set"))
if not set_files:
    raise RuntimeError("Nessun .set trovato.")

raw = read_eeglab_robust(set_files[0])
print(raw)
print("sfreq:", raw.info["sfreq"], "nchan:", raw.info["nchan"], "dur(s):", raw.n_times/raw.info["sfreq"])


RuntimeError: Il .set NON risulta HDF5, ma SciPy/MNE lo interpreta come MAT v4 e fallisce.
Probabile: file corrotto/download incompleto oppure .set non-EEGLAB reale.
Errore originale: Mat 4 mopt wrong format, byteswapping problem?

In [11]:
# ============================
# NEUROCORE — EEGLAB .set RUN (robusto) — IRCCS-ready
# ============================

from pathlib import Path
import re, time, json
import numpy as np
import pandas as pd

import mne
from mne.time_frequency import psd_array_welch

# -------- CONFIG --------
DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")
OUT_DIR = DATASET_DIR / "NEUROCORE_ANALYSIS_IRCCS"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESAMPLE_HZ = 250
HP_HZ, LP_HZ = 1.0, 40.0
NOTCH_HZ = 50
REF = "average"

WIN_S = 4.0
STEP_S = 1.0
N_STATES = 4
BANDS = {"theta": (4, 8), "alpha": (8, 12)}

t0 = time.time()

# -------- utils --------
def parse_bids_entities(name: str):
    ent = {}
    for k, v in re.findall(r"([a-zA-Z]+)-([a-zA-Z0-9]+)", name):
        ent[k.lower()] = v
    return ent

def shannon_entropy(p_vec, eps=1e-12):
    p = np.asarray(p_vec, dtype=float)
    s = p.sum()
    if not np.isfinite(s) or s <= 0:
        return np.nan
    p = p / (s + eps)
    p = np.clip(p, eps, 1.0)
    return float(-(p * np.log(p)).sum())

def kmeans_simple(X, k, n_iter=40, seed=42):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    k = int(min(max(1, k), n))
    C = X[rng.choice(n, size=k, replace=False)].copy()
    for _ in range(n_iter):
        d2 = ((X[:, None, :] - C[None, :, :])**2).sum(axis=2)
        lab = d2.argmin(axis=1)
        C_new = C.copy()
        for j in range(k):
            sel = (lab == j)
            if sel.any():
                C_new[j] = X[sel].mean(axis=0)
        if np.allclose(C, C_new, atol=1e-6):
            break
        C = C_new
    return lab, C

def sliding_windows(n_samples, sfreq, win_s, step_s):
    win = int(round(win_s * sfreq))
    step = int(round(step_s * sfreq))
    if win <= 0 or step <= 0:
        return np.array([], dtype=int), win
    starts = np.arange(0, max(0, n_samples - win + 1), step, dtype=int)
    return starts, win

def bandpower_from_psd(psd, freqs, band):
    fmin, fmax = band
    m = (freqs >= fmin) & (freqs < fmax)
    if not np.any(m):
        return np.zeros(psd.shape[0], dtype=float)
    df = float(np.mean(np.diff(freqs)))
    return psd[:, m].sum(axis=1) * df

def needs_fdt(set_path: Path) -> bool:
    # euristica: se esiste .fdt accanto -> lo consideriamo richiesto/atteso
    return set_path.with_suffix(".fdt").exists()

def read_eeglab_safe(set_path: Path):
    # tentativo singolo, senza patch strane: ora il problema era solo "._"
    return mne.io.read_raw_eeglab(set_path, preload=True, verbose="ERROR")

# -------- find inputs (NO Apple ._ files) --------
set_files = sorted(
    p for p in DATASET_DIR.rglob("*.set")
    if not p.name.startswith("._")
)

if not set_files:
    raise RuntimeError("Nessun .set reale trovato (attenzione: escludiamo i file ._).")

pd.DataFrame({"set_file": [p.relative_to(DATASET_DIR).as_posix() for p in set_files]}).to_csv(
    OUT_DIR / "inputs_set_files.csv", index=False
)

# -------- run --------
per_file_rows, per_window_rows, errors_rows = [], [], []

for set_path in set_files:
    rel = set_path.relative_to(DATASET_DIR).as_posix()
    ent = parse_bids_entities(set_path.name)
    subject = ent.get("sub", "")
    ses = ent.get("ses", "")
    task = ent.get("task", "")
    run = ent.get("run", "")

    try:
        # --- read ---
        raw = read_eeglab_safe(set_path)

        # --- pick channels ---
        # preferisci EEG, ma tieni EOG/ECG/EMG se presenti per compatibilità (poi analisi su EEG)
        raw.pick_types(eeg=True, eog=True, ecg=True, emg=True, stim=False, misc=True)

        # --- reference ---
        if "eeg" in raw:
            raw.set_eeg_reference(REF, projection=False, verbose="ERROR")

        # --- filters ---
        try:
            raw.notch_filter(NOTCH_HZ, verbose="ERROR")
        except Exception:
            pass
        raw.filter(HP_HZ, LP_HZ, verbose="ERROR")
        raw.resample(RESAMPLE_HZ, npad="auto", verbose="ERROR")

        eeg = raw.copy().pick("eeg")
        data = eeg.get_data()
        if data.size == 0:
            raise RuntimeError("Nessun canale EEG dopo pick().")

        sfreq = float(eeg.info["sfreq"])
        n_ch, n_samp = data.shape
        duration_s = n_samp / sfreq

        # --- PSD globale ---
        psd, freqs = psd_array_welch(
            data, sfreq=sfreq, fmin=HP_HZ, fmax=LP_HZ,
            n_fft=int(round(sfreq * 4)), n_overlap=int(round(sfreq * 2)),
            verbose=False
        )
        total_power = psd.sum(axis=1)
        total_norm = total_power / (total_power.sum() + 1e-12)

        alpha = bandpower_from_psd(psd, freqs, BANDS["alpha"])
        global_alpha_ratio = float(alpha.sum() / (total_power.sum() + 1e-12))
        spatial_entropy_total_power = shannon_entropy(total_norm)

        # --- dinamica finestre ---
        starts, win = sliding_windows(n_samp, sfreq, WIN_S, STEP_S)

        if len(starts) < 3:
            per_file_rows.append({
                "relative_path": rel, "subject": subject, "session": ses, "task": task, "run": run,
                "n_channels_eeg": int(n_ch), "sfreq_hz": sfreq, "duration_s": round(duration_s, 3),
                "global_alpha_ratio": global_alpha_ratio,
                "spatial_entropy_total_power": spatial_entropy_total_power,
                "stabilita_organizzazione_funzione_media": np.nan,
                "transizioni_stato_n": np.nan,
                "persistenza_temporale_mediana_s": np.nan,
                "indeterminatezza_funzione_entropy": np.nan,
            })
            continue

        X, win_times = [], []
        for s0 in starts:
            seg = data[:, s0:s0+win]
            psd_w, freqs_w = psd_array_welch(
                seg, sfreq=sfreq, fmin=HP_HZ, fmax=LP_HZ,
                n_fft=int(round(sfreq * 2)), n_overlap=int(round(sfreq * 1)),
                verbose=False
            )
            theta_w = bandpower_from_psd(psd_w, freqs_w, BANDS["theta"])
            alpha_w = bandpower_from_psd(psd_w, freqs_w, BANDS["alpha"])
            topo = (theta_w + alpha_w)
            topo = topo / (topo.sum() + 1e-12)
            X.append(topo)
            win_times.append(s0 / sfreq)

        X = np.vstack(X)
        win_times = np.asarray(win_times)

        labels, _ = kmeans_simple(X, k=N_STATES, n_iter=40, seed=42)

        # stabilità: correlazione topo tra finestre consecutive
        corr_seq = []
        for a in range(len(X) - 1):
            c = np.corrcoef(X[a], X[a+1])[0, 1]
            corr_seq.append(c if np.isfinite(c) else np.nan)
        corr_seq = np.asarray(corr_seq)
        stabilita_media = float(np.nanmean(corr_seq))

        # transizioni
        transizioni = int(np.sum(labels[1:] != labels[:-1]))

        # persistenza (run-length * STEP)
        runs, rlen = [], 1
        for a in range(1, len(labels)):
            if labels[a] == labels[a-1]:
                rlen += 1
            else:
                runs.append(rlen); rlen = 1
        runs.append(rlen)
        runs = np.asarray(runs, dtype=float)
        persistenza_mediana_s = float(np.median(runs) * STEP_S)

        # indeterminatezza: entropia occupazione stati
        occ = np.bincount(labels, minlength=int(labels.max())+1).astype(float)
        indet_entropy = shannon_entropy(occ)

        # audit windows
        topo_sim_next = np.r_[corr_seq, np.nan]
        for t_sec, lab, cval in zip(win_times, labels, topo_sim_next):
            per_window_rows.append({
                "relative_path": rel, "subject": subject, "session": ses, "task": task, "run": run,
                "t_window_start_s": round(float(t_sec), 3),
                "state_label": int(lab),
                "topo_similarity_next": float(cval) if np.isfinite(cval) else np.nan
            })

        per_file_rows.append({
            "relative_path": rel, "subject": subject, "session": ses, "task": task, "run": run,
            "n_channels_eeg": int(n_ch), "sfreq_hz": sfreq, "duration_s": round(duration_s, 3),
            "global_alpha_ratio": global_alpha_ratio,
            "spatial_entropy_total_power": spatial_entropy_total_power,
            "stabilita_organizzazione_funzione_media": stabilita_media,
            "transizioni_stato_n": transizioni,
            "persistenza_temporale_mediana_s": persistenza_mediana_s,
            "indeterminatezza_funzione_entropy": indet_entropy,
        })

    except Exception as e:
        errors_rows.append({
            "relative_path": rel,
            "error": repr(e),
            "hint": "Se errori su FDT: verifica .fdt accanto al .set e integrità del dataset. Se canali EEG=0: file non EEG o metadata."
        })

# -------- export --------
df_files = pd.DataFrame(per_file_rows)
df_windows = pd.DataFrame(per_window_rows)
df_errors = pd.DataFrame(errors_rows)

df_files.to_csv(OUT_DIR / "NEUROCORE_metrics_per_file.csv", index=False)
df_windows.to_csv(OUT_DIR / "NEUROCORE_state_dynamics_windows.csv", index=False)
df_errors.to_csv(OUT_DIR / "NEUROCORE_errors.csv", index=False)

if not df_files.empty and "subject" in df_files.columns:
    df_sub = (df_files.groupby("subject", dropna=False)
              .agg(
                  n_recordings=("relative_path", "count"),
                  duration_s_mean=("duration_s", "mean"),
                  stabilita_organizzazione_funzione_media=("stabilita_organizzazione_funzione_media", "mean"),
                  transizioni_stato_n=("transizioni_stato_n", "sum"),
                  persistenza_temporale_mediana_s=("persistenza_temporale_mediana_s", "median"),
                  indeterminatezza_funzione_entropy=("indeterminatezza_funzione_entropy", "mean"),
                  global_alpha_ratio=("global_alpha_ratio", "mean"),
              )
              .reset_index())
else:
    df_sub = pd.DataFrame()

df_sub.to_csv(OUT_DIR / "NEUROCORE_summary_by_subject.csv", index=False)

elapsed = time.time() - t0
report = "\n".join([
    "NEUROCORE — Report analisi EEG (EEGLAB .set) — IRCCS-ready",
    "=" * 72,
    f"Directory dataset: {DATASET_DIR}",
    f"Directory output:  {OUT_DIR}",
    "",
    "Output prodotti",
    "-" * 72,
    "1) NEUROCORE_metrics_per_file.csv        — metriche per registrazione",
    "2) NEUROCORE_state_dynamics_windows.csv  — dinamica finestre/stati (audit trail)",
    "3) NEUROCORE_summary_by_subject.csv      — sintesi per soggetto",
    "4) NEUROCORE_errors.csv                  — eventuali errori per-file",
    "",
    f"File .set trovati (esclusi ._): {len(set_files)}",
    f"File analizzati OK: {len(df_files)}",
    f"File con errori: {len(df_errors)}",
    f"Tempo esecuzione: {elapsed:.2f} secondi",
])
(OUT_DIR / "REPORT_ANALYSIS_IRCCS.txt").write_text(report, encoding="utf-8")

print(report)
display(df_files.head(30))
display(df_sub.head(30))
display(df_errors.head(30))

NEUROCORE — Report analisi EEG (EEGLAB .set) — IRCCS-ready
Directory dataset: /Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia
Directory output:  /Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia/NEUROCORE_ANALYSIS_IRCCS

Output prodotti
------------------------------------------------------------------------
1) NEUROCORE_metrics_per_file.csv        — metriche per registrazione
2) NEUROCORE_state_dynamics_windows.csv  — dinamica finestre/stati (audit trail)
3) NEUROCORE_summary_by_subject.csv      — sintesi per soggetto
4) NEUROCORE_errors.csv                  — eventuali errori per-file

File .set trovati (esclusi ._): 14
File analizzati OK: 0
File con errori: 14
Tempo esecuzione: 51.92 secondi


""


""


,relative_path,error,hint
0,sub-G01/ses-1/eeg/sub-G01_ses-1_task-PictureNa...,"TypeError(""'NoneType' object is not iterable"")",Se errori su FDT: verifica .fdt accanto al .se...
1,sub-G01/ses-12/eeg/sub-G01_ses-12_task-Picture...,"TypeError(""'NoneType' object is not iterable"")",Se errori su FDT: verifica .fdt accanto al .se...
2,sub-G01/ses-2/eeg/sub-G01_ses-2_task-PictureNa...,"TypeError(""'NoneType' object is not iterable"")",Se errori su FDT: verifica .fdt accanto al .se...
3,sub-G01/ses-20/eeg/sub-G01_ses-20_task-Picture...,"TypeError(""'NoneType' object is not iterable"")",Se errori su FDT: verifica .fdt accanto al .se...
4,sub-G01/ses-4/eeg/sub-G01_ses-4_task-PictureNa...,"TypeError(""'NoneType' object is not iterable"")",Se errori su FDT: verifica .fdt accanto al .se...
5,sub-G01/ses-6/eeg/sub-G01_ses-6_task-PictureNa...,"TypeError(""'NoneType' object is not iterable"")",Se errori su FDT: verifica .fdt accanto al .se...
6,sub-G01/ses-8/eeg/sub-G01_ses-8_task-PictureNa...,"TypeError(""'NoneType' object is not iterable"")",Se errori su FDT: verifica .fdt accanto al .se...
7,sub-G02/ses-1/eeg/sub-G02_ses-1_task-PictureNa...,"TypeError(""'NoneType' object is not iterable"")",Se errori su FDT: verifica .fdt accanto al .se...
8,sub-G02/ses-12/eeg/sub-G02_ses-12_task-Picture...,"TypeError(""'NoneType' object is not iterable"")",Se errori su FDT: verifica .fdt accanto al .se...
9,sub-G02/ses-2/eeg/sub-G02_ses-2_task-PictureNa...,"TypeError(""'NoneType' object is not iterable"")",Se errori su FDT: verifica .fdt accanto al .se...


In [12]:
from pathlib import Path

DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")

missing = []

for s in DATASET_DIR.rglob("*.set"):
    if s.name.startswith("._"):
        continue
    fdt = s.with_suffix(".fdt")
    if not fdt.exists():
        missing.append(str(s))

print("SET senza FDT:", len(missing))
missing[:5]

SET senza FDT: 0


[]

In [13]:
import mne

def read_eeglab_safe(set_path):
    # 1) prova standard MA senza eventi->annotations (bypassa il crash 'NoneType not iterable')
    try:
        return mne.io.read_raw_eeglab(
            set_path,
            preload=True,
            events_as_annotations=False,
            verbose="ERROR"
        )
    except TypeError:
        # 2) fallback: alcune versioni hanno firma diversa o bug intermittente
        return mne.io.read_raw_eeglab(
            set_path,
            preload=True,
            verbose="ERROR"
        )


In [14]:
import mne

# ---- PATCH: rendi iterabile 'None' dove MNE legge gli eventi EEGLAB ----
import mne.io.eeglab.eeglab as _eeglab_mod

if hasattr(_eeglab_mod, "_read_annotations_eeglab"):
    _ORIG = _eeglab_mod._read_annotations_eeglab
    def _read_annotations_eeglab_compat(*args, **kwargs):
        ann = _ORIG(*args, **kwargs)
        # se la funzione ritorna None in qualche corner case, trasformiamo in "nessuna annotation"
        return ann if ann is not None else []
    _eeglab_mod._read_annotations_eeglab = _read_annotations_eeglab_compat

def read_eeglab_safe(set_path):
    return mne.io.read_raw_eeglab(
        set_path,
        preload=True,
        events_as_annotations=False,
        verbose="ERROR"
    )


In [15]:
# ============================
# NEUROCORE — EEGLAB .set RUN (fallback robusto su .fdt) — IRCCS-ready
# ============================

from pathlib import Path
import re, time, traceback
import numpy as np
import pandas as pd

import mne
from mne.time_frequency import psd_array_welch
import scipy.io

# -------- CONFIG --------
DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")
OUT_DIR = DATASET_DIR / "NEUROCORE_ANALYSIS_IRCCS"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESAMPLE_HZ = 250
HP_HZ, LP_HZ = 1.0, 40.0
NOTCH_HZ = 50
REF = "average"

WIN_S = 4.0
STEP_S = 1.0
N_STATES = 4
BANDS = {"theta": (4, 8), "alpha": (8, 12)}

t0 = time.time()

# -------- utils --------
def parse_bids_entities(name: str):
    ent = {}
    for k, v in re.findall(r"([a-zA-Z]+)-([a-zA-Z0-9]+)", name):
        ent[k.lower()] = v
    return ent

def shannon_entropy(p_vec, eps=1e-12):
    p = np.asarray(p_vec, dtype=float)
    s = p.sum()
    if not np.isfinite(s) or s <= 0:
        return np.nan
    p = p / (s + eps)
    p = np.clip(p, eps, 1.0)
    return float(-(p * np.log(p)).sum())

def kmeans_simple(X, k, n_iter=40, seed=42):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    k = int(min(max(1, k), n))
    C = X[rng.choice(n, size=k, replace=False)].copy()
    for _ in range(n_iter):
        d2 = ((X[:, None, :] - C[None, :, :])**2).sum(axis=2)
        lab = d2.argmin(axis=1)
        C_new = C.copy()
        for j in range(k):
            sel = (lab == j)
            if sel.any():
                C_new[j] = X[sel].mean(axis=0)
        if np.allclose(C, C_new, atol=1e-6):
            break
        C = C_new
    return lab, C

def sliding_windows(n_samples, sfreq, win_s, step_s):
    win = int(round(win_s * sfreq))
    step = int(round(step_s * sfreq))
    if win <= 0 or step <= 0:
        return np.array([], dtype=int), win
    starts = np.arange(0, max(0, n_samples - win + 1), step, dtype=int)
    return starts, win

def bandpower_from_psd(psd, freqs, band):
    fmin, fmax = band
    m = (freqs >= fmin) & (freqs < fmax)
    if not np.any(m):
        return np.zeros(psd.shape[0], dtype=float)
    df = float(np.mean(np.diff(freqs)))
    return psd[:, m].sum(axis=1) * df

def _safe_str(x):
    if x is None:
        return ""
    if isinstance(x, bytes):
        try:
            return x.decode("utf-8", errors="ignore")
        except Exception:
            return ""
    return str(x)

def load_eeglab_via_mat_fdt(set_path: Path) -> mne.io.Raw:
    """
    Loader robusto EEGLAB:
    - legge .set (MAT v5) con scipy.io.loadmat
    - se EEG.data è path/filename -> legge .fdt float32 little-endian
    - costruisce RawArray senza eventi (evita crash 'NoneType not iterable')
    """
    md = scipy.io.loadmat(set_path, struct_as_record=False, squeeze_me=True)
    if "EEG" not in md:
        raise RuntimeError("Variabile 'EEG' non trovata nel .set (non è un EEGLAB .set standard).")

    EEG = md["EEG"]

    # srate / dimensioni
    srate = float(getattr(EEG, "srate", np.nan))
    nbchan = int(getattr(EEG, "nbchan", 0))
    pnts = int(getattr(EEG, "pnts", 0))
    trials = int(getattr(EEG, "trials", 1))
    if not np.isfinite(srate) or srate <= 0:
        raise RuntimeError("srate non valido nel .set.")
    if nbchan <= 0 or pnts <= 0:
        raise RuntimeError("nbchan/pnts non validi nel .set.")

    # channel labels
    ch_names = None
    chanlocs = getattr(EEG, "chanlocs", None)
    if chanlocs is not None:
        try:
            # chanlocs può essere array di struct o singolo struct
            if isinstance(chanlocs, np.ndarray):
                labs = []
                for c in chanlocs.flatten():
                    lab = _safe_str(getattr(c, "labels", ""))
                    labs.append(lab if lab else f"EEG{len(labs)+1:03d}")
                ch_names = labs
            else:
                # singolo canale
                lab = _safe_str(getattr(chanlocs, "labels", "")) or "EEG001"
                ch_names = [lab]
        except Exception:
            ch_names = None
    if not ch_names or len(ch_names) != nbchan:
        ch_names = [f"EEG{c+1:03d}" for c in range(nbchan)]

    # data
    data_field = getattr(EEG, "data", None)
    if data_field is None:
        raise RuntimeError("EEG.data è None nel .set.")

    # Caso 1: data già numerico nel .set
    if isinstance(data_field, np.ndarray) and np.issubdtype(data_field.dtype, np.number):
        dat = np.asarray(data_field, dtype=np.float64)
        # EEGLAB può essere (nbchan, pnts) o (nbchan, pnts, trials)
        if dat.ndim == 2:
            if dat.shape[0] != nbchan:
                dat = dat.T
            if dat.shape[0] != nbchan:
                raise RuntimeError(f"Shape inattesa EEG.data: {dat.shape} (nbchan atteso {nbchan}).")
            data_2d = dat
        elif dat.ndim == 3:
            # concat trials nel tempo
            if dat.shape[0] != nbchan:
                # tentativo permute
                dat = np.transpose(dat, (1, 0, 2))
            if dat.shape[0] != nbchan:
                raise RuntimeError(f"Shape inattesa EEG.data (3D): {dat.shape} (nbchan atteso {nbchan}).")
            data_2d = dat.reshape(nbchan, dat.shape[1]*dat.shape[2], order="F")
        else:
            raise RuntimeError(f"EEG.data ha ndim={dat.ndim} non gestito.")
    else:
        # Caso 2: EEG.data è filename o path del .fdt
        fdt_name = _safe_str(data_field).strip()
        if not fdt_name:
            # fallback: spesso si chiama come il .set
            fdt_path = set_path.with_suffix(".fdt")
        else:
            fdt_path = (set_path.parent / fdt_name)
            if not fdt_path.exists():
                # prova solo basename
                fdt_path2 = set_path.with_suffix(".fdt")
                fdt_path = fdt_path2

        if not fdt_path.exists():
            raise RuntimeError(f".fdt non trovato: atteso {fdt_path}")

        # EEGLAB .fdt tipico: float32 little-endian, vettore EEG.data(:) in ordine MATLAB (column-major)
        expected = nbchan * pnts * max(1, trials)
        x = np.fromfile(fdt_path, dtype="<f4", count=expected)
        if x.size != expected:
            raise RuntimeError(f"Lettura .fdt incompleta: letti {x.size} float32, attesi {expected}.")

        if trials == 1:
            dat3 = x.reshape((nbchan, pnts), order="F")
            data_2d = dat3.astype(np.float64, copy=False)
        else:
            dat3 = x.reshape((nbchan, pnts, trials), order="F")
            data_2d = dat3.reshape((nbchan, pnts*trials), order="F").astype(np.float64, copy=False)

    info = mne.create_info(ch_names=ch_names, sfreq=srate, ch_types=["eeg"] * nbchan)
    raw = mne.io.RawArray(data_2d, info, verbose="ERROR")
    return raw

def read_eeglab_ultra_robust(set_path: Path) -> mne.io.Raw:
    # 1) prova MNE disabilitando conversione eventi->annotations (bypass crash)
    try:
        return mne.io.read_raw_eeglab(set_path, preload=True, events_as_annotations=False, verbose="ERROR")
    except Exception:
        # 2) fallback: loader custom (mat+fdt)
        return load_eeglab_via_mat_fdt(set_path)

# -------- find inputs (NO Apple ._ files) --------
set_files = sorted(
    p for p in DATASET_DIR.rglob("*.set")
    if not p.name.startswith("._")
)

if not set_files:
    raise RuntimeError("Nessun .set reale trovato (esclusi i file ._).")

pd.DataFrame({"set_file": [p.relative_to(DATASET_DIR).as_posix() for p in set_files]}).to_csv(
    OUT_DIR / "inputs_set_files.csv", index=False
)

# -------- run --------
per_file_rows, per_window_rows, errors_rows = [], [], []

for set_path in set_files:
    rel = set_path.relative_to(DATASET_DIR).as_posix()
    ent = parse_bids_entities(set_path.name)
    subject = ent.get("sub", "")
    ses = ent.get("ses", "")
    task = ent.get("task", "")
    run = ent.get("run", "")

    try:
        raw = read_eeglab_ultra_robust(set_path)

        # Preprocessing (solo EEG)
        raw.pick_types(eeg=True)
        if len(raw.ch_names) == 0:
            raise RuntimeError("Nessun canale EEG disponibile.")
        raw.set_eeg_reference(REF, projection=False, verbose="ERROR")

        try:
            raw.notch_filter(NOTCH_HZ, verbose="ERROR")
        except Exception:
            pass
        raw.filter(HP_HZ, LP_HZ, verbose="ERROR")
        raw.resample(RESAMPLE_HZ, npad="auto", verbose="ERROR")

        data = raw.get_data()
        sfreq = float(raw.info["sfreq"])
        n_ch, n_samp = data.shape
        duration_s = n_samp / sfreq

        # PSD globale
        psd, freqs = psd_array_welch(
            data, sfreq=sfreq, fmin=HP_HZ, fmax=LP_HZ,
            n_fft=int(round(sfreq * 4)), n_overlap=int(round(sfreq * 2)),
            verbose=False
        )
        total_power = psd.sum(axis=1)
        total_norm = total_power / (total_power.sum() + 1e-12)

        alpha = bandpower_from_psd(psd, freqs, BANDS["alpha"])
        global_alpha_ratio = float(alpha.sum() / (total_power.sum() + 1e-12))
        spatial_entropy_total_power = shannon_entropy(total_norm)

        # Dinamica finestre
        starts, win = sliding_windows(n_samp, sfreq, WIN_S, STEP_S)
        if len(starts) < 3:
            per_file_rows.append({
                "relative_path": rel, "subject": subject, "session": ses, "task": task, "run": run,
                "n_channels_eeg": int(n_ch), "sfreq_hz": sfreq, "duration_s": round(duration_s, 3),
                "global_alpha_ratio": global_alpha_ratio,
                "spatial_entropy_total_power": spatial_entropy_total_power,
                "stabilita_organizzazione_funzione_media": np.nan,
                "transizioni_stato_n": np.nan,
                "persistenza_temporale_mediana_s": np.nan,
                "indeterminatezza_funzione_entropy": np.nan,
            })
            continue

        X, win_times = [], []
        for s0 in starts:
            seg = data[:, s0:s0+win]
            psd_w, freqs_w = psd_array_welch(
                seg, sfreq=sfreq, fmin=HP_HZ, fmax=LP_HZ,
                n_fft=int(round(sfreq * 2)), n_overlap=int(round(sfreq * 1)),
                verbose=False
            )
            theta_w = bandpower_from_psd(psd_w, freqs_w, BANDS["theta"])
            alpha_w = bandpower_from_psd(psd_w, freqs_w, BANDS["alpha"])
            topo = (theta_w + alpha_w)
            topo = topo / (topo.sum() + 1e-12)
            X.append(topo)
            win_times.append(s0 / sfreq)

        X = np.vstack(X)
        win_times = np.asarray(win_times)

        labels, _ = kmeans_simple(X, k=N_STATES, n_iter=40, seed=42)

        corr_seq = []
        for a in range(len(X) - 1):
            c = np.corrcoef(X[a], X[a+1])[0, 1]
            corr_seq.append(c if np.isfinite(c) else np.nan)
        corr_seq = np.asarray(corr_seq)
        stabilita_media = float(np.nanmean(corr_seq))

        transizioni = int(np.sum(labels[1:] != labels[:-1]))

        runs, rlen = [], 1
        for a in range(1, len(labels)):
            if labels[a] == labels[a-1]:
                rlen += 1
            else:
                runs.append(rlen); rlen = 1
        runs.append(rlen)
        runs = np.asarray(runs, dtype=float)
        persistenza_mediana_s = float(np.median(runs) * STEP_S)

        occ = np.bincount(labels, minlength=int(labels.max())+1).astype(float)
        indet_entropy = shannon_entropy(occ)

        topo_sim_next = np.r_[corr_seq, np.nan]
        for t_sec, lab, cval in zip(win_times, labels, topo_sim_next):
            per_window_rows.append({
                "relative_path": rel, "subject": subject, "session": ses, "task": task, "run": run,
                "t_window_start_s": round(float(t_sec), 3),
                "state_label": int(lab),
                "topo_similarity_next": float(cval) if np.isfinite(cval) else np.nan
            })

        per_file_rows.append({
            "relative_path": rel, "subject": subject, "session": ses, "task": task, "run": run,
            "n_channels_eeg": int(n_ch), "sfreq_hz": sfreq, "duration_s": round(duration_s, 3),
            "global_alpha_ratio": global_alpha_ratio,
            "spatial_entropy_total_power": spatial_entropy_total_power,
            "stabilita_organizzazione_funzione_media": stabilita_media,
            "transizioni_stato_n": transizioni,
            "persistenza_temporale_mediana_s": persistenza_mediana_s,
            "indeterminatezza_funzione_entropy": indet_entropy,
        })

    except Exception as e:
        errors_rows.append({
            "relative_path": rel,
            "error": repr(e),
            "traceback_head": "\n".join(traceback.format_exc().splitlines()[:12])
        })

# -------- export --------
df_files = pd.DataFrame(per_file_rows)
df_windows = pd.DataFrame(per_window_rows)
df_errors = pd.DataFrame(errors_rows)

df_files.to_csv(OUT_DIR / "NEUROCORE_metrics_per_file.csv", index=False)
df_windows.to_csv(OUT_DIR / "NEUROCORE_state_dynamics_windows.csv", index=False)
df_errors.to_csv(OUT_DIR / "NEUROCORE_errors.csv", index=False)

if not df_files.empty and "subject" in df_files.columns:
    df_sub = (df_files.groupby("subject", dropna=False)
              .agg(
                  n_recordings=("relative_path", "count"),
                  duration_s_mean=("duration_s", "mean"),
                  stabilita_organizzazione_funzione_media=("stabilita_organizzazione_funzione_media", "mean"),
                  transizioni_stato_n=("transizioni_stato_n", "sum"),
                  persistenza_temporale_mediana_s=("persistenza_temporale_mediana_s", "median"),
                  indeterminatezza_funzione_entropy=("indeterminatezza_funzione_entropy", "mean"),
                  global_alpha_ratio=("global_alpha_ratio", "mean"),
              )
              .reset_index())
else:
    df_sub = pd.DataFrame()

df_sub.to_csv(OUT_DIR / "NEUROCORE_summary_by_subject.csv", index=False)

elapsed = time.time() - t0
report = "\n".join([
    "NEUROCORE — Report analisi EEG (EEGLAB .set) — IRCCS-ready",
    "=" * 72,
    f"Directory dataset: {DATASET_DIR}",
    f"Directory output:  {OUT_DIR}",
    "",
    "Output prodotti",
    "-" * 72,
    "1) NEUROCORE_metrics_per_file.csv        — metriche per registrazione",
    "2) NEUROCORE_state_dynamics_windows.csv  — dinamica finestre/stati (audit trail)",
    "3) NEUROCORE_summary_by_subject.csv      — sintesi per soggetto",
    "4) NEUROCORE_errors.csv                  — eventuali errori per-file (con traceback_head)",
    "",
    f"File .set trovati (esclusi ._): {len(set_files)}",
    f"File analizzati OK: {len(df_files)}",
    f"File con errori: {len(df_errors)}",
    f"Tempo esecuzione: {elapsed:.2f} secondi",
])
(OUT_DIR / "REPORT_ANALYSIS_IRCCS.txt").write_text(report, encoding="utf-8")

print(report)
display(df_files.head(30))
display(df_sub.head(30))
display(df_errors.head(30))

NEUROCORE — Report analisi EEG (EEGLAB .set) — IRCCS-ready
Directory dataset: /Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia
Directory output:  /Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia/NEUROCORE_ANALYSIS_IRCCS

Output prodotti
------------------------------------------------------------------------
1) NEUROCORE_metrics_per_file.csv        — metriche per registrazione
2) NEUROCORE_state_dynamics_windows.csv  — dinamica finestre/stati (audit trail)
3) NEUROCORE_summary_by_subject.csv      — sintesi per soggetto
4) NEUROCORE_errors.csv                  — eventuali errori per-file (con traceback_head)

File .set trovati (esclusi ._): 14
File analizzati OK: 0
File con errori: 14
Tempo esecuzione: 2.26 secondi


""


""


,relative_path,error,traceback_head
0,sub-G01/ses-1/eeg/sub-G01_ses-1_task-PictureNa...,"RuntimeError(""Variabile 'EEG' non trovata nel ...","Traceback (most recent call last):\n File ""/v..."
1,sub-G01/ses-12/eeg/sub-G01_ses-12_task-Picture...,"RuntimeError(""Variabile 'EEG' non trovata nel ...","Traceback (most recent call last):\n File ""/v..."
2,sub-G01/ses-2/eeg/sub-G01_ses-2_task-PictureNa...,"RuntimeError(""Variabile 'EEG' non trovata nel ...","Traceback (most recent call last):\n File ""/v..."
3,sub-G01/ses-20/eeg/sub-G01_ses-20_task-Picture...,"RuntimeError(""Variabile 'EEG' non trovata nel ...","Traceback (most recent call last):\n File ""/v..."
4,sub-G01/ses-4/eeg/sub-G01_ses-4_task-PictureNa...,"RuntimeError(""Variabile 'EEG' non trovata nel ...","Traceback (most recent call last):\n File ""/v..."
5,sub-G01/ses-6/eeg/sub-G01_ses-6_task-PictureNa...,"RuntimeError(""Variabile 'EEG' non trovata nel ...","Traceback (most recent call last):\n File ""/v..."
6,sub-G01/ses-8/eeg/sub-G01_ses-8_task-PictureNa...,"RuntimeError(""Variabile 'EEG' non trovata nel ...","Traceback (most recent call last):\n File ""/v..."
7,sub-G02/ses-1/eeg/sub-G02_ses-1_task-PictureNa...,"RuntimeError(""Variabile 'EEG' non trovata nel ...","Traceback (most recent call last):\n File ""/v..."
8,sub-G02/ses-12/eeg/sub-G02_ses-12_task-Picture...,"RuntimeError(""Variabile 'EEG' non trovata nel ...","Traceback (most recent call last):\n File ""/v..."
9,sub-G02/ses-2/eeg/sub-G02_ses-2_task-PictureNa...,"RuntimeError(""Variabile 'EEG' non trovata nel ...","Traceback (most recent call last):\n File ""/v..."


In [16]:
!pip install mne-bids -q

In [17]:
from mne_bids import BIDSPath, read_raw_bids
from pathlib import Path

DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")

raws = []

for eeg_json in DATASET_DIR.rglob("*_eeg.json"):
    bp = BIDSPath(root=DATASET_DIR)
    bp.update(
        subject=eeg_json.name.split("_")[0].split("-")[1],
        session=eeg_json.name.split("_")[1].split("-")[1],
        task=eeg_json.name.split("_")[2].split("-")[1],
        suffix="eeg",
        extension=".set"
    )

    try:
        raw = read_raw_bids(bp, verbose=False)
        raws.append(raw)
        print("OK:", bp.basename)
    except Exception as e:
        print("FAIL:", bp.basename, e)

print("\nTotali caricati:", len(raws))


IndexError: list index out of range

In [18]:
from mne_bids import BIDSPath, read_raw_bids, get_entity_vals
from pathlib import Path

DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")

subjects = get_entity_vals(DATASET_DIR, "subject")
sessions = get_entity_vals(DATASET_DIR, "session")

raws = []

for sub in subjects:
    for ses in sessions:
        try:
            bp = BIDSPath(
                subject=sub,
                session=ses,
                suffix="eeg",
                extension=".set",
                root=DATASET_DIR
            )

            raw = read_raw_bids(bp, verbose=False)
            raws.append(raw)

            print("OK →", bp.basename)

        except Exception:
            pass

print("\nTotali caricati:", len(raws))


Totali caricati: 0


In [19]:
from pathlib import Path
import pandas as pd

from mne_bids import BIDSPath, read_raw_bids, find_matching_paths

DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")
OUT_DIR = DATASET_DIR / "NEUROCORE_ANALYSIS_IRCCS"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1) Trova TUTTI i file EEG BIDS che corrispondono (non indovinare subject/session/task dal nome)
paths = find_matching_paths(
    root=DATASET_DIR,
    datatype="eeg",
    suffix="eeg",
    extension=".set"
)

print("File BIDS trovati:", len(paths))
if len(paths) > 0:
    print("Esempio:", paths[0])

raws = []
rows = []

# 2) Prova a caricarli uno a uno e registra errori
for bp in paths:
    try:
        raw = read_raw_bids(bp, verbose=False)
        raws.append(raw)
        rows.append({"basename": bp.basename, "status": "OK", "error": ""})
        print("OK →", bp.basename)
    except Exception as e:
        rows.append({"basename": bp.basename, "status": "FAIL", "error": repr(e)})

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "BIDS_read_log.csv", index=False)

print("\nTotali caricati:", len(raws))
print("Log:", OUT_DIR / "BIDS_read_log.csv")

# 3) Se 0, mostra i primi errori (diagnosi immediata)
if len(raws) == 0 and not df.empty:
    print("\nPrime 10 righe FAIL:")
    display(df[df["status"]=="FAIL"].head(10))

TypeError: find_matching_paths() got an unexpected keyword argument 'datatype'

In [20]:
from pathlib import Path
from mne_bids import BIDSPath, read_raw_bids

DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")

raws = []

# trova tutti i .set EEG reali
set_files = [f for f in DATASET_DIR.rglob("*_eeg.set") if not f.name.startswith("._")]

print("File EEG trovati:", len(set_files))

for f in set_files:
    parts = f.parts

    # estrai subject e session dalla struttura BIDS (robusto)
    subject = [p for p in parts if p.startswith("sub-")][0].split("-")[1]
    session = [p for p in parts if p.startswith("ses-")][0].split("-")[1]

    try:
        bp = BIDSPath(
            subject=subject,
            session=session,
            suffix="eeg",
            extension=".set",
            root=DATASET_DIR
        )

        raw = read_raw_bids(bp, verbose=False)
        raws.append(raw)

        print("OK →", bp.basename)

    except Exception as e:
        print("FAIL →", f.name, "|", e)

print("\nTotale caricati:", len(raws))

File EEG trovati: 14
FAIL → sub-G01_ses-1_task-PictureNaming_run-1_eeg.set | "bids_path" must contain `root`, `subject`, and `task` attributes but it's missing `task`.
FAIL → sub-G01_ses-12_task-PictureNaming_run-12_eeg.set | "bids_path" must contain `root`, `subject`, and `task` attributes but it's missing `task`.
FAIL → sub-G01_ses-2_task-PictureNaming_run-2_eeg.set | "bids_path" must contain `root`, `subject`, and `task` attributes but it's missing `task`.
FAIL → sub-G01_ses-20_task-PictureNaming_run-20_eeg.set | "bids_path" must contain `root`, `subject`, and `task` attributes but it's missing `task`.
FAIL → sub-G01_ses-4_task-PictureNaming_run-4_eeg.set | "bids_path" must contain `root`, `subject`, and `task` attributes but it's missing `task`.
FAIL → sub-G01_ses-6_task-PictureNaming_run-6_eeg.set | "bids_path" must contain `root`, `subject`, and `task` attributes but it's missing `task`.
FAIL → sub-G01_ses-8_task-PictureNaming_run-8_eeg.set | "bids_path" must contain `root`, `sub

In [21]:
from pathlib import Path
from mne_bids import BIDSPath, read_raw_bids

DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")

raws = []

set_files = [f for f in DATASET_DIR.rglob("*_eeg.set") if not f.name.startswith("._")]

print("File EEG trovati:", len(set_files))

for f in set_files:
    name = f.name

    subject = [p for p in f.parts if p.startswith("sub-")][0].split("-")[1]
    session = [p for p in f.parts if p.startswith("ses-")][0].split("-")[1]

    task = name.split("task-")[1].split("_")[0]

    run = None
    if "run-" in name:
        run = name.split("run-")[1].split("_")[0]

    try:
        bp = BIDSPath(
            subject=subject,
            session=session,
            task=task,
            run=run,
            suffix="eeg",
            extension=".set",
            root=DATASET_DIR
        )

        raw = read_raw_bids(bp, verbose=False)
        raws.append(raw)

        print("OK →", bp.basename)

    except Exception as e:
        print("FAIL →", name, "|", e)

print("\nTotale caricati:", len(raws))

File EEG trovati: 14
FAIL → sub-G01_ses-1_task-PictureNaming_run-1_eeg.set | 'NoneType' object is not iterable
FAIL → sub-G01_ses-12_task-PictureNaming_run-12_eeg.set | 'NoneType' object is not iterable
FAIL → sub-G01_ses-2_task-PictureNaming_run-2_eeg.set | 'NoneType' object is not iterable
FAIL → sub-G01_ses-20_task-PictureNaming_run-20_eeg.set | 'NoneType' object is not iterable
FAIL → sub-G01_ses-4_task-PictureNaming_run-4_eeg.set | 'NoneType' object is not iterable
FAIL → sub-G01_ses-6_task-PictureNaming_run-6_eeg.set | 'NoneType' object is not iterable
FAIL → sub-G01_ses-8_task-PictureNaming_run-8_eeg.set | 'NoneType' object is not iterable
FAIL → sub-G02_ses-1_task-PictureNaming_run-1_eeg.set | 'NoneType' object is not iterable
FAIL → sub-G02_ses-12_task-PictureNaming_run-12_eeg.set | 'NoneType' object is not iterable
FAIL → sub-G02_ses-2_task-PictureNaming_run-2_eeg.set | 'NoneType' object is not iterable
FAIL → sub-G02_ses-20_task-PictureNaming_run-20_eeg.set | 'NoneType' obje

In [22]:
from pathlib import Path
import mne

DATASET_DIR = Path("/Volumes/KINGSTON/04920_Ecosystem/DATASETS/tACS for Patients with Post-Stroke Anomia")

raws = []

set_files = [f for f in DATASET_DIR.rglob("*_eeg.set") if not f.name.startswith("._")]

print("File EEG trovati:", len(set_files))

for f in set_files:
    try:
        raw = mne.io.read_raw_eeglab(
            f,
            preload=True,
            events_as_annotations=False,  # ← evita crash
            verbose=False
        )

        raws.append(raw)
        print("OK →", f.name)

    except Exception as e:
        print("FAIL →", f.name, "|", e)

print("\nTotale caricati:", len(raws))

File EEG trovati: 14
FAIL → sub-G01_ses-1_task-PictureNaming_run-1_eeg.set | read_raw_eeglab() got an unexpected keyword argument 'events_as_annotations'
FAIL → sub-G01_ses-12_task-PictureNaming_run-12_eeg.set | read_raw_eeglab() got an unexpected keyword argument 'events_as_annotations'
FAIL → sub-G01_ses-2_task-PictureNaming_run-2_eeg.set | read_raw_eeglab() got an unexpected keyword argument 'events_as_annotations'
FAIL → sub-G01_ses-20_task-PictureNaming_run-20_eeg.set | read_raw_eeglab() got an unexpected keyword argument 'events_as_annotations'
FAIL → sub-G01_ses-4_task-PictureNaming_run-4_eeg.set | read_raw_eeglab() got an unexpected keyword argument 'events_as_annotations'
FAIL → sub-G01_ses-6_task-PictureNaming_run-6_eeg.set | read_raw_eeglab() got an unexpected keyword argument 'events_as_annotations'
FAIL → sub-G01_ses-8_task-PictureNaming_run-8_eeg.set | read_raw_eeglab() got an unexpected keyword argument 'events_as_annotations'
FAIL → sub-G02_ses-1_task-PictureNaming_run-